# Podcast Emotion and Content Analysis

This project explores the linguistic, emotional, and topical characteristics of podcast episodes using Natural Language Processing (NLP) methods.  
By combining pre-trained language models and structured metadata, we aim to uncover the emotional tone, thematic categories, and audience alignment of each podcast episode.

Our pipeline operates at two complementary levels:
- **Turn-level analysis:** Emotion detection and safety tagging for individual speaker turns.
- **Episode-level analysis:** Aggregation of features to classify episodes into genres, estimate audience interests, and identify potentially unsafe content.


### Motivation

Podcasts are a rich and fast-growing medium, with content ranging from personal storytelling to political commentary.  
However, the scale and diversity of podcast data make manual analysis impractical.  

Automatic understanding of podcast episodes can support:
- **Listeners**, by improving recommendation and discovery systems.
- **Producers**, by providing insights into emotional tone and audience engagement.
- **Platforms**, by enabling automated content moderation and brand safety filtering.

Despite advances in text-based NLP, podcasts present unique challenges: conversational structure, speaker variability, and diverse topics.  
This project seeks to address these challenges by building an interpretable and scalable NLP pipeline tailored for podcast data.


### Project Goals

The primary objectives of this project are:

1. **Turn-level emotion detection** — Identify emotions (e.g., joy, anger, sadness) in each speaker turn using a pre-trained emotion recognition model.
2. **Episode-level categorization** — Classify each episode into broad categories such as *Technology*, *Comedy*, or *Politics*.
3. **Brand safety detection** — Flag episodes that contain unsafe or sensitive content, including hate speech, violence, or adult themes.
4. **Audience alignment prediction** — Estimate the likely audience or interest group best aligned with each episode based on its emotional and linguistic profile.

Collectively, these tasks aim to produce an interpretable summary of each episode — showing how speakers’ emotions, topics, and tone interact across the conversation.


### Expected Outcomes

By the end of this project, we aim to:
- Deliver a fully documented pipeline capable of processing large-scale podcast transcript data.
- Demonstrate emotion and safety predictions on speaker turns using transformer-based models.
- Aggregate turn-level features to classify and describe entire episodes.
- Provide interpretable summaries for each episode showing emotion distribution, category prediction, and safety assessment.

This proof of concept validates that pre-trained NLP models can be combined and scaled to perform fine-grained, multi-level podcast analysis.


## 1. Data Loading and Overview

To start, we'll load the sample datasets, `episodeLevelDataSample` and `speakerTurnDataSample`. The full datasets are over 20 GB, so we're using these smaller versions for the demonstration in this milestone.

In [ ]:
import gzip
import json
import pandas as pd

episode_level = []
with gzip.open("data/episodeLevelDataSample.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        episode_level.append(json.loads(line))

episode_df = pd.DataFrame(episode_level)

In [ ]:
speakerTurn = []
with gzip.open("data/speakerTurnDataSample.jsonl.gz", "rt", encoding="utf-8") as f:
    for line in f:
        speakerTurn.append(json.loads(line))

speaker_df = pd.DataFrame(speakerTurn)

### Columns and Examples

#### Episode Level

The `episodeLevelDataSample` dataset provides metadata for each podcast episode in the sample, including information like the episode title, description, podcast title, and categories. The mp3url in this dataset is particularly important as it acts as a unique identifier for each episode, allowing us to link it with the corresponding speaker turn data later on. We'll use this data to understand the context of the speaker turns and potentially for tasks like episode classification and analysis based on episode-level features.

In [ ]:
print("episodeLevelDataSample columns: ", episode_df.keys())

print("\n episodeLevelDataSample examples: ")
episode_df.sample(5) # Display 5 random samples from the dataset

#### Speaker Turn Level

The `speakerTurnDataSample` dataset contains information about individual speaker turns within each podcast episode. This includes the most important one which is the actual text spoken (`turnText`), the identified speaker, the start and end times of the turn, and the mp3url to link it back to the episode metadata in the episodeLevelDataSample.

In [ ]:
print("speakerTurnDataSample columns: ", speaker_df.keys())

print("\n speakerTurnDataSample examples: ")
speaker_df.sample(5) # Display 5 random samples from the dataset

Given the large number of columns in these datasets, we want to select a subset of columns from each that are relevant to our analysis and discard the rest.

## 2. Data Preprocessing

In this section, we perform a series of preprocessing steps to prepare the dataset for downstream analysis and modeling.

The primary objective is to retain only the most relevant information from the raw `speakerTurnData` and `episodeLevelData` sources while ensuring that textual fields — particularly `turnText` — are clean, standardized, and ready for the tasks.

This involves:

* Selecting key columns that capture speaker and episode metadata.
* Cleaning and normalizing text data to remove noise (e.g., music markers, punctuation, excess whitespace).
* Filtering out irrelevant or empty content.
* Aggregating speaker turns and merging with episode-level information for a unified dataset.

### 2.1 Select Relevant Columns (Speaker-Turn Data)
We extract a subset of columns from the original dataset to create a compact version containing the most informative features for each speaker turn.

| Column                  | Description                                                |
| :---------------------- | :--------------------------------------------------------- |
| **turnText**            | Transcribed text of the speaker’s turn                     |
| **speaker**             | Identifier for the speaker                                 |
| **mp3url**              | Link to the episode (useful for grouping turns by episode) |
| **turnCount**           | Sequential position of the turn within an episode          |
| **inferredSpeakerRole** | Predicted speaker role (e.g., host, guest)                 |
| **startTime**           | Start time of a turn                                       |
| **endTime**             | End time of a turn                                         |


In [ ]:
cols = ['turnText', 'speaker', 'mp3url', 'turnCount', 'inferredSpeakerRole',  'startTime', 'endTime']
speaker_df1 = pd.DataFrame(speakerTurn)[cols].copy()

speaker_df1[:20]

In [ ]:
print(f"Total number of turns: {len(speaker_df1)}")
average_text_length = speaker_df1['turnText'].str.len().mean()
print(f"Average turn text length: {average_text_length:.2f}")
average_word_length = speaker_df1['turnText'].apply(lambda x: len(str(x).split())).mean()
print(f"Average number of words: {average_word_length:.2f}")

#### 2.1.1 Distribution of Turn Text Lengths

This histogram shows the distribution of the length of each speaker turn in characters. A logarithmic scale is used on the x-axis to accommodate the wide range of turn lengths, including some very long turns. Understanding turn length distribution is important for selecting appropriate NLP models and processing strategies.

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram of the turn text lengths
turn_lengths = speaker_df1['turnText'].str.len().tolist()
plt.figure(figsize=(10, 6))
plt.hist(turn_lengths, bins=100, edgecolor='black')
plt.xlabel('Turn Text Length')
plt.ylabel('Frequency')
plt.yscale('log')
plt.title('Distribution of Turn Text Lengths')
plt.show()

#### 2.1.2 Top 20 of most common length turns

In [ ]:
from collections import Counter

length_counts = Counter(turn_lengths)

for length, count in length_counts.most_common(20):
    print(f"Length {length}: {count} turns")

### 2.2 Text Normalization

We standardize the text to lowercase and normalize whitespace.
This step ensures consistent tokenization and reduces vocabulary redundancy (e.g., “The” and “the” are treated the same).

In [ ]:
print(speaker_df1['turnText'][0])

speaker_df1['filtered_text'] = (
    speaker_df1['turnText']
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(speaker_df1['filtered_text'][0])

### 2.3 Noise Removal

The transcripts contain several types of non-linguistic artifacts such as musical cues, blank-audio markers, ellipses, and transcription symbols.

These tokens provide no semantic value and can bias subsequent NLP analysis.

To obtain cleaner text representations, we remove all non-speech elements and normalize leftover formatting.

In [ ]:
import re

# --- 1. Define stronger unwanted patterns ---
# Matches variations like:
# [music], (music playing), [background music fades], music:, blank_audio etc.
unwanted_patterns = [
    r'\[.*?music.*?\]',       # [music], [background music], [music playing]
    r'\(.*?music.*?\)',       # (music), (music playing), (background music)
    r'\[.*?blank_audio.*?\]', # [blank_audio] or variations
    r'\(.*?blank_audio.*?\)', # (blank_audio)
    r'\bmusic\b',             # standalone 'music'
    r'\bblank_audio\b'        # standalone 'blank_audio'
]

# --- 2. Apply removal ---
for pattern in unwanted_patterns:
    speaker_df1['filtered_text'] = speaker_df1['filtered_text'].str.replace(pattern, '', regex=True)

# --- 3. Clean malformed or stray brackets ---
def clean_brackets(text):
    """Removes stray [ or ] while preserving valid bracketed expressions."""
    return re.sub(r'(\[[^\[\]]+\])|[\[\]]', lambda m: m.group(1) or '', text)

speaker_df1['filtered_text'] = speaker_df1['filtered_text'].apply(clean_brackets)

# --- 4. Remove conversation markers or ellipses ---
speaker_df1['filtered_text'] = (
    speaker_df1['filtered_text']
    .str.replace('>>', '', regex=False)
    .str.replace(r'\.\.\.', '', regex=True)  # remove ...
)

# --- 4. Remove near-empty or symbol-only entries ---
speaker_df1 = speaker_df1[speaker_df1['filtered_text'].str.len() > 1]
speaker_df1 = speaker_df1[~speaker_df1['filtered_text'].str.match(r'^\W*$')]

# --- 6. Final whitespace normalization ---
speaker_df1['filtered_text'] = speaker_df1['filtered_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

speaker_df1[['turnText','filtered_text']][:20]


### 2.4 Token-Level and Structural Cleaning

At this stage, we refine the cleaned transcripts to make them suitable for modeling.
This involves removing punctuation and stopwords, merging consecutive turns from the same speaker, and filtering out low-information text segments.

#### 2.4.1 Text Standardization and Punctuation Removal

We define a text-cleaning function that lowercases the text, trims whitespace, and optionally removes punctuation.
This ensures consistency in tokenization and reduces vocabulary fragmentation

In [ ]:
import string

def clean_text(text, remove_punct=True):
    """
    Lowercase, trim, and optionally remove punctuation.
    Handles missing or non-string entries safely.
    """
    if isinstance(text, str):
        if remove_punct:
            text = text.translate(str.maketrans('', '', string.punctuation))
        return text.lower().strip()
    return text

# Apply the cleaning function
speaker_df1['filtered_text'] = speaker_df1['filtered_text'].apply(clean_text)

speaker_df1[['turnText','filtered_text']][:20]


#### 2.4.2 Merge Consecutive Turns by the Same Speaker

Some transcripts split a speaker’s speech into multiple turns even when uninterrupted.
To maintain coherent speaker contributions, we merge consecutive turns by the same speaker into single entries.

In [ ]:
print(f'Number of turns before merging: {len(speaker_df1)}')

# Ensure the speaker field is a string
speaker_df1['speaker'] = speaker_df1['speaker'].apply(
    lambda x: ','.join(x) if isinstance(x, list) else str(x)
)

# Create a group index that increments when the speaker changes
speaker_df1['group'] = (speaker_df1['speaker'] != speaker_df1['speaker'].shift()).cumsum()

# Merge consecutive text segments by the same speaker
merged_df = (
    speaker_df1
    .groupby(['group', 'speaker'], as_index=False)
    .agg({
        'turnText': lambda x: ' '.join(x),
        'filtered_text': lambda x: ' '.join(x),      # concatenate text
        'startTime': 'first',                   # earliest start time
        'endTime': 'last',                      # latest end time
        'mp3url': 'first',                      # keep the episode link
        'inferredSpeakerRole': 'first',         # role stays the same for the speaker
    })
)

# Turn it back as a list
# Turn it back as a list
merged_df['speaker'] = merged_df['speaker'].apply(
    lambda x: [s.strip() for s in x.split(',')] if isinstance(x, str) else x
)

# Drop the temporary grouping column if not needed
speaker_df1 = merged_df.drop(columns=['group'], errors='ignore')

print(f'Number of turns after merging: {len(speaker_df1)}')

speaker_df1[['speaker','filtered_text']][:20]


#### 2.4.3 Stopword Removal

Stopwords (e.g., the, is, and) are common words that typically carry little semantic meaning.

Removing them reduces noise and emphasizes content-bearing terms.

In [ ]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Remove stopwords from each turn
speaker_df1['filtered_text'] = speaker_df1['filtered_text'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stop_words])
)

speaker_df1[:20]

#### 2.4.4 Remove Extremely Short or Low-Information Turns

Short turns (single words such as “yeah”, “okay”, or “right”) often add noise to topic or embedding models.
We remove these to focus the analysis on meaningful utterances.

In [ ]:
# Keep only turns with more than one word
old_len = len(speaker_df1)
speaker_df1 = speaker_df1[speaker_df1['filtered_text'].apply(lambda x: len(str(x).split()) > 1)]

print(f'Number of turns before cleanup: {old_len}')
print(f"Total number of turns: {len(speaker_df1)}")
average_text_length = speaker_df1['filtered_text'].str.len().mean()
print(f"Average turn text length: {average_text_length:.2f}")
average_word_length = speaker_df1['filtered_text'].apply(lambda x: len(str(x).split())).mean()
print(f"Average number of words: {average_word_length:.2f}")

##### 2.4.4.1 Distribution of Turn Text Lengths after cleanup

In [ ]:
import matplotlib.pyplot as plt

# Plot a histogram of the turn text lengths
turn_lengths = speaker_df1['filtered_text'].str.len().tolist()
plt.figure(figsize=(10, 6))
plt.hist(turn_lengths, bins=100, edgecolor='black')
plt.xlabel('Turn Text Length')
plt.ylabel('Frequency')
plt.yscale('log')
plt.title('Distribution of Turn Text Lengths')
plt.show()

##### 2.4.4.2 Top 20 of most common length turns

In [ ]:
from collections import Counter

length_counts = Counter(turn_lengths)

for length, count in length_counts.most_common(20):
    print(f"Length {length}: {count} turns")

#### 2.4.5 Final Output

In [ ]:
for index, row in speaker_df1.iloc[420:425].iterrows():
    print(f"Speaker  : {row['speaker']}")
    print(f"Original : {row['turnText'][:100]}... ({len(row['turnText'])} length)",)
    print(f"Filtered : {row['filtered_text'][:100]}... ({len(row['filtered_text'])} length)",)
    print()

### 2.5 Episode-Level Preprocessing

#### 2.4.1 Select Relevant Columns
We extract a subset of columns from the original `episodeLevelData` DataFrame, focusing on those that provide contextual and categorical information about each episode.

| Column                           | Description                                                   |
| :------------------------------- | :------------------------------------------------------------ |
| **epTitle**, **epDescription**   | Episode title and description                                 |
| **mp3url**                       | Unique episode identifier (used to link with speaker turns)   |
| **podTitle**, **podDescription** | Podcast-level metadata for context and categorization         |
| **explicit**                     | Flag indicating whether the episode contains explicit content |
| **category1–category10**         | Topical categories associated with the episode                |



In [ ]:
episode_df1 = episode_df[[
    'epTitle', 'mp3url', 'podTitle', 'explicit',
    'category1', 'category2', 'category3', 'category4', 'category5',
    'category6', 'category7', 'category8', 'category9', 'category10'
]].copy()

episode_df1[:20]


In [ ]:
# How many unique episodes?
unique_episodes_count = len(episode_df1['mp3url'].unique())
print(f"Number of unique episodes: {unique_episodes_count}")


# How many different categories in total?
category_columns = [f'category{i}' for i in range(1, 9)]
all_categories = episode_df1[category_columns].values.flatten()
unique_categories = pd.Series(all_categories).dropna().unique()
total_unique_categories = len(unique_categories)
print(f"\nTotal number of unique categories: {total_unique_categories}")


#### 2.4.2 Handle Missing and Sparse Fields

We first check for missing values in key columns.

Given the dataset’s large size, a small number of missing entries can be safely dropped without compromising representativeness

In [ ]:
print("\nMissing values in episode_df1:")
print(episode_df1[['epTitle', 'mp3url', 'podTitle', 'category1']].isnull().sum())

category_columns = [f'category{i}' for i in range(1, 11)]
unique_episodes_count = len(episode_df1['mp3url'].unique())

print("\nNumber of episodes with each category:")
for col in category_columns:
    count = episode_df1[col].notnull().sum()
    print(f"{col}: {count}/{unique_episodes_count}")


In [ ]:
# What categories are represented and any imbalance?
# We can look at the distribution of the first category as a starting point
print("\nDistribution of Category1:")
print('Number of unique categories: ', len(episode_df1['category1'].value_counts()))
print(episode_df1['category1'].value_counts())

print("\nDistribution of Category2:")
print('Number of unique categories: ', len(episode_df1['category2'].value_counts()))
print(episode_df1['category2'].value_counts())

print("\nDistribution of Category3:")
print('Number of unique categories: ', len(episode_df1['category3'].value_counts()))
print(episode_df1['category3'].value_counts())

In [ ]:
# Check for missing values in key columns
print("\nMissing values in episode_df1:")
print(episode_df1[['epTitle', 'mp3url', 'podTitle', 'category1']].isnull().sum())

Most episodes include only a few categories, with all values in `category9` and `category10` being empty.

We therefore remove these two columns to reduce sparsity and simplify the dataset.

In [ ]:
print(f"Shape of episode_df1 before dropping missing values and columns: {episode_df1.shape}")

def drop_missing_values_and_columns_episodelvl(episode_dataframe):
    """
    Drop rows missing key metadata and remove unused category columns.
    """
    # Drop rows missing critical fields
    episode_dataframe1 = episode_dataframe.dropna(subset=['podTitle', 'category1'])

    # Drop category9 and category10 if they exist
    columns_to_drop = ['category9', 'category10']
    existing_columns_to_drop = [c for c in columns_to_drop if c in episode_dataframe1.columns]
    if existing_columns_to_drop:
        episode_dataframe1 = episode_dataframe1.drop(columns=existing_columns_to_drop)
    return episode_dataframe1

# Apply the cleanup
episode_df1 = drop_missing_values_and_columns_episodelvl(episode_df1)

print(f"Shape of episode_df1 after dropping missing values and columns: {episode_df1.shape}")


### 2.6 Merge TurnText - and Episode-Level Data

At this stage, we combine the cleaned speaker-turn and episode-level data to form a unified dataset at the episode level.

This step allows us to analyze or model episodes holistically — using both metadata and spoken content.

#### 2.6.1 Aggregate Speaker Turns per Episode

We first group all turnText entries by mp3url (the episode identifier) and concatenate them into a single text block for each episode.

In [ ]:
# Group speaker turns by episode (mp3url) and concatenate the turnText
episode_text_df = (
    speaker_df1
    .groupby('mp3url')
    .apply(lambda df: [
        {
            "speaker": str(row['speaker']),
            "turnText": str(row['filtered_text']),
            # optional: include timing info if available
            **({"startTime": row['startTime'], "endTime": row['endTime']} 
               if 'startTime' in df.columns and 'endTime' in df.columns else {})
        }
        for _, row in df.iterrows()
    ])
    .reset_index(name='turns_list')
)

episode_text_df[:5]


#### 2.6.2 Merge Turns with Episode

We then merge the aggregated text with `episode_df1` using `mp3url` as the join key.

In [ ]:
# Merge the combined text with episode-level dataframe
episode_df1 = pd.merge(episode_df1, episode_text_df, on='mp3url', how='left')
print(f'Number of episodes before cleanup: {len(episode_df1)}')

# Keep only episodes that actually exist in speaker_df1
valid_urls = set(speaker_df1['mp3url'])
episode_df1 = episode_df1[episode_df1['mp3url'].isin(valid_urls)].reset_index(drop=True)
print(f'Number of episodes after cleanup: {len(episode_df1)}')

episode_df1[:5]

This results in one row per episode, containing:

* metadata (e.g., title, categories),
* and the entire spoken transcript as turns_list.

##### 2.6.2.1 Distribution of Turns per Episode
This histogram shows the distribution of the number of speaker turns within each podcast episode. This helps us understand the typical conversational structure and density of the episodes in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Calculate the number of turns per episode
turns_per_episode_counts = episode_df1['turns_list'].apply(len).tolist()

# Plot a histogram of the number of turns per episode
plt.figure(figsize=(10, 6))
plt.hist(turns_per_episode_counts, bins=50, edgecolor='black')
plt.title('Distribution of Turns per Episode')
plt.xlabel('Number of Turns')
plt.ylabel('Number of Episodes')
plt.xlim(0, 2400)
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# How many turns per episode?
turns_per_episode = episode_df1['turns_list'].apply(len).mean()
print(f"Average turns per episode: {turns_per_episode:.2f}")

# How many speakers per episode?
# The 'speaker' column appears to be lists of speaker labels.
def count_unique_speakers_list(speaker_list):
    # Ensure the input is a list before counting unique speakers
    if isinstance(speaker_list, list):
        return len(set(speaker_list))
    else:
        # Print the problematic entry and its type if it's not a list
        print(f"Unexpected type in speaker column: {speaker_list}, Type: {type(speaker_list)}")
        return 0 # Handle cases where the entry is not a list

speaker_df1['num_speakers'] = speaker_df1['speaker'].apply(count_unique_speakers_list)
speakers_per_episode = speaker_df1.groupby('mp3url')['num_speakers'].max().mean()
print(f"Average speakers per episode: {speakers_per_episode:.2f}")

*Note: The y-axis is logarithmic.*

An interesting observation is that the **vast majority** of `turnText` segments are quite short. This suggests that in episodes featuring multiple speakers, frequent interruptions or rapid exchanges may lead to very brief interactions. This pattern is important to keep in mind when feeding the emotion model, as an abundance of short turns could distort the overall emotion distribution at the episode level.

On the other hand, there are also a number of turns **exceeding 10,000 characters**, which are likely outliers caused by transcription or merging artifacts. These unusually long turns can also bias model performance and will require a separate cleaning or truncation strategy in later preprocessing steps. **(TO DO)**


#### 2.6.3 Add Episode Title to Speaker Level Dataset

We can also add the episode titles to speakerLevel dataset, which will come in handy later.

In [ ]:
# Merge speaker_df1 with episode_df1 to get the episode title
speaker_df1 = pd.merge(speaker_df1, episode_df1[['mp3url', 'epTitle']], on='mp3url', how='left')

# Display the first few rows to verify the new column
display(speaker_df1.head())

#### 2.6.4 Extract and Summarize Categories

Finally, we extract the unique set of categories across all category columns to understand the topical diversity of the dataset.

In [ ]:
# Extract all category columns
category_columns = [f'category{i}' for i in range(1, 9)]

# Flatten and get unique non-null category values
all_categories = (
    pd.Series(episode_df1[category_columns].values.flatten())
    .dropna()
    .unique()
    .tolist()
)

print(f"Total number of unique categories found: {len(all_categories)}")
print("All unique categories:", all_categories)


##### 2.6.4.1 Distribution of Top 20 Most Frequent Categories (all categories)

Here we present the **top 20 most frequent categories**, aggregated across all category fields (`category1` to `category8`). This overview provides a clear picture of the dataset’s thematic landscape, highlighting the dominant topics and revealing potential areas for more granular or targeted analysis.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
all_categories_series = pd.Series(episode_df1[category_columns].values.flatten()).dropna()

# Get the top 20 most frequent categories
top_20_categories = all_categories_series.value_counts().head(20)

# Create a bar chart of the top 20 categories
plt.figure(figsize=(10, 6))
top_20_categories.plot(kind='bar')
plt.title('Top 20 Most Frequent Categories (from all categories)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

##### 2.6.4.2 Distribution of Top 20 Most Frequent Categories (from category1)

Here we show the **top 20 most frequent categories in `category1`**. This provides insights into the primary genres and main topics covered by the podcasts in the dataset. Analyzing the distribution of the primary category is crucial for understanding the core thematic composition of the data and for informing decisions about episode-level classification tasks.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
all_categories = episode_df1['category1'].values.flatten()
all_categories_series = pd.Series(all_categories).dropna()

# Get the top 20 most frequent categories
top_20_categories = all_categories_series.value_counts().head(20)

# Create a bar chart of the top 20 categories
plt.figure(figsize=(10, 6))
top_20_categories.plot(kind='bar')
plt.title('Top 20 Most Frequent Categories (from category1)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

##### 2.6.4.3 Distribution of Top 20 Most Frequent Categories (from category2)

And here we show the top 20 most frequent categories in category2. Examining this distribution helps reveal common secondary themes or sub-genres present in the podcast episodes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Combine all category columns into a single Series
all_categories = episode_df1['category2'].values.flatten()
all_categories_series = pd.Series(all_categories).dropna()

# Get the top 20 most frequent categories
top_20_categories = all_categories_series.value_counts().head(20)

# Create a bar chart of the top 20 categories
plt.figure(figsize=(10, 6))
top_20_categories.plot(kind='bar')
plt.title('Top 20 Most Frequent Categories (from category2)')
plt.xlabel('Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 2.6.5 Final Results 

In [ ]:
for i in range(10, 15):
    ep_title = episode_df1.loc[i, 'epTitle']
    mp3_url = episode_df1.loc[i, 'mp3url']
    turns = episode_df1.loc[i, 'turns_list']
    categories = episode_df1.loc[i, category_columns]
    category_list = [cat for cat in categories if cat is not None]


    print(f"\n{'='*60}")
    print(f"Episode {i+1}: {ep_title}")
    print(f"MP3 URL   : {mp3_url}")
    print(f"Categories: {category_list}")
    print(f"Turns     : {len(turns)}")
    print(f"{'-'*60}")

    for t in turns[:8]:  # show first 8 turns
        preview = t['turnText'][:100].replace('\n', ' ') + ('...' if len(t['turnText']) > 100 else '')
        print(f"[{t['speaker']}] {preview}")
    if len(turns) > 8:
        print(f"... ({len(turns) - 8} more turns)")
    print(f"{'='*60}\n")


## 3. Split Datasets

In this section, we split the dataset into **training** and **validation** sets.

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(episode_df1, test_size=0.2, random_state=42)

print("Split Overview")
print("-" * 40)
print(f"Total episodes     : {len(episode_df1)}")
print(f"Train episodes     : {len(train)}")
print(f"Validation episodes: {len(val)}\n")


## 4. Next Steps (Proof of Concept) 

### 4.1 Emotional Detector  (Early Demonstration)

In [ ]:
import torch

device = 'cpu'
if torch.backends.mps.is_available() : device = 'mps'
if torch.cuda.is_available(): device = 'cuda'

print("Using device: ", device)

In [ ]:
#from transformers.utils.hub import move_cache
#move_cache()

In [ ]:
from transformers import pipeline


df_sample = episode_df1.dropna(subset=['turns_list']).sample(30, random_state=41)  # tiny sample
texts = [turn['filtered_text'] for turns in df_sample['turns_list'] for turn in turns if 'filtered_text' in turn]

# Load emotion detection model
emotion_classifier = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device=device)

batch_size = 8  

batch_results = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    batch_preds = emotion_classifier(batch, truncation=True)
    batch_results.extend(batch_preds)

# Check one result
print(batch_results[0])

# Add results to dataframe
df_sample['emotion'] = [r[0]['label'] for r in batch_results]
df_sample['emotion_score'] = [r[0]['score'] for r in batch_results]


#### 4.1.1 Aggregate emotions per episode

By aggregating emotion predictions across all speaker turns within each episode, we can understand the overall emotional landscape of a conversation. For each episode, we calculate the proportion of turns associated with different emotions, providing insight into the emotional dynamics throughout the discussion. This aggregation helps reveal patterns like emotional progression, dominant moods, or significant emotional shifts during the episode.

Note: As this small demo only showcases 30 random turnTexts for 30 random episodes, the distribution will only show one emotion for each episode, but this is just to give some insights to what we aim for.

In [ ]:
# Group by episode (mp3url) - group by episode name instead somehow
emotion_counts = (
    df_sample.groupby(['epTitle', 'emotion'])
    .size()
    .unstack(fill_value=0)
)

emotion_ratios = emotion_counts.div(emotion_counts.sum(axis=1), axis=0)
emotion_ratios.head(10)


### 4.1.2 Average Emotion Distribution

The emotion distribution below shows that 'neutral' is indeed the most frequent classification, which aligns with the nature of podcast conversations - many turns consist of short, functional utterances even after filtering, that carry little emotional content. While the model can detect a range of emotions (anger, disgust, fear, joy, neutral, sadness, surprise), neutral dominates the distribution due to these brief conversational elements. This suggests that for meaningful emotional analysis, we might want to focus on longer turns or aggregate emotions across conversation segments rather than individual short turns.

In [ ]:
import matplotlib.pyplot as plt

emotion_ratios.mean().sort_values().plot(kind='barh')
plt.title("Average Emotion Distribution (sample of 30 turns)")
plt.xlabel("Proportion")
plt.ylabel("Emotion")
plt.show()

#### 4.1.3 Emotional Graph (TO DO)

In [ ]:
# TO DO

### 4.2 Category Classification

This section contains only the initial data-preparation steps for a future episode-level category classification task. It does NOT include model training or evaluation. The purpose here is to produce a clean, ready dataset for later modeling.

Contents:
- Sample selection & label extraction: filter episodes with non-null `combined_turnText` and `category1`; optionally restrict to the top‑N frequent categories to keep the sample manageable; extract and encode the primary category as the target vector.
- Text vectorization (TF‑IDF): fit a `TfidfVectorizer` (unigrams + bigrams, basic stopword filtering, and an optional max feature/min document frequency) and transform the selected episode texts into sparse TF‑IDF features.

Result: a prepared feature matrix (`X_tfidf`) and label vector (`y`) stored for the next assignment where we will implement model training, evaluation, and error analysis.

#### 4.2.1 Sample Selection and Label Extraction

Here we'll:
1. Load episode data and select relevant columns
2. Keep only episodes with non-null text and category
3. Focus on top-K most frequent categories
4. Create feature matrix X and target vector y

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

# Load episode data (assuming we have episode_df1 from previous notebook)
# Select rows with non-null text and category
df = episode_df1[['epTitle', 'combined_turnText', 'category1']].dropna()

# Keep only top 20 most frequent categories
top_k = 20
top_categories = df['category1'].value_counts().nlargest(top_k).index
df = df[df['category1'].isin(top_categories)]

# Create feature matrix X and target vector y
X = df['combined_turnText']
y = df['category1']

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)

print("Dataset shape:", df.shape)
print("\nCategory distribution:")
for i, category in enumerate(le.classes_):
    count = (y == i).sum()
    print(f"{category}: {count} episodes ({count/len(y):.1%})")

#### 4.2.2 Text Vectorization (TF-IDF)

Convert the text data into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency). This:
- Captures word importance through term frequency and document frequency
- Handles common words appropriately through IDF weighting
- Creates a sparse feature matrix suitable for classification

In [ ]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=10000,  # Limit vocabulary size
    min_df=5,           # Ignore terms that appear in < 5 documents
    ngram_range=(1, 2), # Use unigrams and bigrams
    stop_words='english'
)

# Fit and transform the text data
X_tfidf = tfidf.fit_transform(X)

print("TF-IDF feature matrix shape:", X_tfidf.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))
print("\nSample features:", list(tfidf.get_feature_names_out())[:10])

#### 4.2.3 Planned model training, evaluation and next steps (summary)



In a later assignment we will build on the prepared features (`X_tfidf`) and labels (`y`) to train and evaluate classification models. High-level plan:

- Train/validation split: stratified 80/20 (optionally hold out a small test set); consider cross-validation for robust estimates.
- Baseline model: Logistic Regression (class_weight='balanced', solver='saga' or 'liblinear') with optional hyperparameter tuning via grid search or randomized CV.
- Metrics: report accuracy, precision/recall/F1 (micro/macro), and per-class support; plot a confusion matrix and calibration curves where relevant.
- Interpretability & error analysis: inspect top features (model coefficients) per class, review misclassified episode excerpts to diagnose label noise or ambiguous content.
- Probabilities & thresholding: show top‑k predicted categories and confidences; consider thresholding for precision-oriented applications.
- Persistence: save vectorizer/tokenizer, label encoder, and trained model (joblib) for later inference.

Future upgrades: experiment with pretrained embeddings or fine-tuned transformers, multi-label setups, and class-rebalancing strategies.

### 4.3 Potential Brand Safety Analysis



This section might explore brand safety analysis of podcast content, though the specific approach and implementation details are still under consideration. Here's a preliminary outline of what could potentially be included:

1. **Content Safety Assessment** (tentative approach)
   - We might analyze text for potentially sensitive or controversial content
   - This could involve identifying mentions of specific topics or themes
   - The scope would need careful definition based on available resources and project requirements

2. **Possible Analysis Methods**
   - We could consider keyword-based detection of sensitive topics
   - Alternatively, we might explore pre-trained content classification models
   - The choice of method would depend on feasibility and accuracy requirements

3. **Potential Challenges**
   - Context interpretation in conversational content
   - Handling nuanced discussions of sensitive topics
   - Balancing detection accuracy with false positives
   - Need for careful consideration of bias in analysis methods

4. **Possible Output Format**
   - If implemented, we might provide episode-level safety ratings
   - Could potentially include segment-level annotations
   - Format and granularity would need to align with practical use cases

Note: This section represents potential future work and the actual implementation may vary significantly based on feasibility analysis and project constraints. The approach to brand safety analysis would need careful consideration of ethical implications and bias mitigation.

### 4.4 Audience Alignment Prediction (TO DO)

This short exploratory section considers ways to estimate which audience segments are likely to engage with an episode using its content signals (topics, tone, and emotion). Based on emotional and topical embeddings, we could apply a zero-shot classification approach to infer likely audience segments (e.g., “young professionals”, “tech enthusiasts”, “general entertainment”). This is an exploratory feasibility step intended to demonstrate the potential of emotion-aware audience profiling rather than a committed methodology; the final approach will depend on available listener data and time constraints.